# 05 분류 모델 확신도 보정 + 처음 보는 음식 감지 + 1·2단계 기준값
**왜:** 06에서 처음 보는 카오소이를 분류 모델이 **치킨 커리 0.61**로 착각 → 2단계로 안 가서 밀을 놓쳤다.
분류 모델은 101종 중 하나를 꼭 골라서 모르는 음식에도 높은 확률을 준다 (과신). 지금 기준(`T_ONE 0.8`, `T_KNOWN 0.3`)은 임시값.

| 단계 | 내용 | 결과 |
|---|---|---|
| ② 확신도 보정 | temperature scaling: 확률을 실제 정답률에 맞게 (Food-101 test 절반으로 맞추고 나머지 절반으로 확인) | 보정 전/후 ECE·NLL 표, 신뢰도 그림 |
| ③ 처음 보는 음식 감지 | 최대 확률(보정 전/후) vs energy 점수. 아는 음식 = 101종 레시피 사진, 처음 보는 음식 = 74종 레시피 사진 (**같은 레시피 사진끼리** 비교해야 공정) | AUROC, FPR@TPR95 표 |
| ⑤ 기준값 | 보정된 1등 확률로 `T_ONE`·`T_KNOWN` 정하기. 74종을 **음식 단위로 반 나눠** 한쪽으로 정하고 다른 쪽으로 확인 | 기준값 + 경로 비율 표 |

- 분류 모델은 다시 학습하지 않는다 (숫자 하나 T만 맞춤). ④ outlier exposure 추가 학습은 이 노트북에 없음
- 결과: 드라이브 `찍먹_실험기록/표/05_*.csv·png`, `best/convnext_tiny/calibration.json` → 숫자를 깃 `modeling/calibration.json`·`knowledge/judge.py`에 반영
- 런타임: GPU 권장 (CPU도 됨, 느림)

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')
OUT, CNX = f'{EXP}/표', f'{EXP}/best/convnext_tiny'
os.makedirs(OUT, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # GPU면 10분쯤 (대부분 Food-101 압축 풀기), CPU면 30분쯤
print('준비 완료', dev)

## 1. 사진 모으기 → 분류 모델 logits

In [ ]:
import json
import numpy as np, pandas as pd
import torch.nn.functional as F
from PIL import Image
from modeling.model import load_model, TF

cnx, classes = load_model(f'{CNX}/best.pt', f'{REPO}/modeling/classes.txt')
cnx = cnx.to(dev)
new_foods = pd.read_csv(f'{REPO}/knowledge/candidates.csv', encoding='utf-8-sig')['class'].tolist()
fake = set(pd.read_csv(f'{IMG}/_fake_md5.csv').md5)

def load(x):
    try:
        return Image.open(io.BytesIO(x) if isinstance(x, bytes) else x).convert('RGB')
    except Exception:
        return None

def recipe_photos(label, n=30):
    """드라이브 레시피 사진 (03에서 모은 것, 가짜 사진 제외) 최대 n장"""
    p = f'{IMG}/{label}.parquet'
    if not os.path.exists(p):
        return []
    d = pd.read_parquet(p, columns=['jpg', 'md5'])
    d = d[~d.md5.isin(fake)]
    return [im for b in d.jpg.sample(min(n, len(d)), random_state=0) if (im := load(b))]

@torch.no_grad()
def logits_of(imgs):
    out = []
    for i in range(0, len(imgs), 128):
        out.append(cnx(torch.stack([TF(im) for im in imgs[i:i + 128]]).to(dev)).float().cpu())
    return torch.cat(out) if out else torch.zeros(0, len(classes))

# Food-101 test (학습에 안 쓴 사진): 음식마다 50장 → 앞 25장 보정용(cal), 뒤 25장 확인용(test)
F101 = '/content/food-101'
if not os.path.exists(f'{F101}/meta/test.txt'):
    subprocess.run(['tar', 'xzf', f'{DATA}/food-101.tar.gz', '-C', '/content'], check=True)
by = {}
for p in open(f'{F101}/meta/test.txt').read().split():
    by.setdefault(p.split('/')[0], []).append(p)
rng = random.Random(0)
cal, tst = [], []
for c in sorted(by):
    s = rng.sample(by[c], 50)
    cal += s[:25]; tst += s[25:]
def f101(paths):
    return logits_of([load(f'{F101}/images/{p}.jpg') for p in paths]), torch.tensor([classes.index(p.split('/')[0]) for p in paths])
Lc, yc = f101(cal)
Lt, yt = f101(tst)

# 레시피 사진: 아는 음식 101종 / 처음 보는 음식 74종 (74종은 음식 단위로 반 나눔: A=기준 정하기, B=확인)
known = {c: logits_of(recipe_photos(c)) for c in classes}
new = {c: logits_of(recipe_photos(c)) for c in new_foods}
new = {c: v for c, v in new.items() if len(v)}
nf = sorted(new); random.Random(1).shuffle(nf)
newA, newB = nf[:len(nf) // 2], nf[len(nf) // 2:]
cat = lambda d, ks: torch.cat([d[k] for k in ks if len(d[k])])
LkA = torch.cat([v[:len(v) // 2] for v in known.values() if len(v)]); ykA = torch.cat([torch.full((len(v) // 2,), classes.index(c)) for c, v in known.items() if len(v)])
LkB = torch.cat([v[len(v) // 2:] for v in known.values() if len(v)]); ykB = torch.cat([torch.full((len(v) - len(v) // 2,), classes.index(c)) for c, v in known.items() if len(v)])
LnA, LnB = cat(new, newA), cat(new, newB)
print(f'Food-101 보정 {len(Lc)} / 확인 {len(Lt)} | 아는 음식 레시피 사진 A {len(LkA)} / B {len(LkB)} | '
      f'처음 보는 음식 {len(new)}종: A {len(newA)}종 {len(LnA)}장 / B {len(newB)}종 {len(LnB)}장')

## 2. 확신도 보정 (temperature scaling)

In [ ]:
def ece(p, y, bins=15):
    conf, pred = p.max(1)
    acc = (pred == y).float()
    e = torch.zeros(())
    for lo in torch.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            e += m.float().mean() * (conf[m].mean() - acc[m].mean()).abs()
    return e.item()

# T 하나만 맞춤 (보정용 절반의 NLL 최소). 분류 모델 가중치는 그대로
logT = torch.zeros(1, requires_grad=True)
opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
def step():
    opt.zero_grad(); l = F.cross_entropy(Lc / logT.exp(), yc); l.backward(); return l
opt.step(step)
T = logT.exp().item()

rows = []
for name, t in [('보정 전 (T=1)', 1.0), (f'보정 후 (T={T:.2f})', T)]:
    p = (Lt / t).softmax(1)
    rows.append({'': name, '정확도': (p.argmax(1) == yt).float().mean().item(), 'ECE ↓': ece(p, yt),
                 'NLL ↓': F.cross_entropy(Lt / t, yt).item(), '평균 1등 확률': p.max(1).values.mean().item()})
res_cal = pd.DataFrame(rows).set_index('').round(4)
res_cal.to_csv(f'{OUT}/05_보정.csv', encoding='utf-8-sig')
display(res_cal)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for t, lab in [(1.0, '보정 전'), (T, '보정 후')]:
    p = (Lt / t).softmax(1); conf, pred = p.max(1); acc = (pred == yt).float()
    edges = torch.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 10: xs.append(conf[m].mean().item()); ys.append(acc[m].mean().item())
    ax[0].plot(xs, ys, 'o-', label=lab)
ax[0].plot([0, 1], [0, 1], 'k--', lw=1); ax[0].set(xlabel='confidence', ylabel='accuracy', title='Reliability (Food-101 test half)'); ax[0].legend()
for L, lab in [(torch.cat([LkA, LkB]), 'known (101, recipe photos)'), (torch.cat([LnA, LnB]), 'unseen (74, recipe photos)')]:
    ax[1].hist((L / T).softmax(1).max(1).values.numpy(), bins=20, range=(0, 1), alpha=0.6, density=True, label=lab)
ax[1].set(xlabel='calibrated top-1 prob', title='Top-1 prob: known vs unseen'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/05_보정_감지.png', dpi=150); plt.show()

## 3. 처음 보는 음식 감지

In [ ]:
from sklearn.metrics import roc_auc_score

SCORES = {   # 클수록 '아는 음식'
    '최대 확률 (보정 전)': lambda L: L.softmax(1).max(1).values,
    '최대 확률 (보정 후)': lambda L: (L / T).softmax(1).max(1).values,
    'energy 점수': lambda L: T * torch.logsumexp(L / T, 1),
}
def ood_row(sk, sn):
    y = np.r_[np.ones(len(sk)), np.zeros(len(sn))]
    s = np.r_[sk.numpy(), sn.numpy()]
    th = np.quantile(sk.numpy(), 0.05)              # 아는 음식 95%를 아는 음식으로 둘 때
    return {'AUROC ↑': roc_auc_score(y, s), 'FPR@TPR95 ↓ (처음 보는 음식을 아는 음식으로 둔 비율)': (sn.numpy() >= th).mean()}

rows = []
for name, f in SCORES.items():
    rows.append({'점수': name, '비교': '레시피 사진: 아는 101종 vs 처음 보는 74종 (공정)',
                 **ood_row(f(torch.cat([LkA, LkB])), f(torch.cat([LnA, LnB])))})
    rows.append({'점수': name, '비교': '참고: Food-101 사진 vs 처음 보는 74종 (사진 출처도 달라 쉬움)',
                 **ood_row(f(Lt), f(torch.cat([LnA, LnB])))})
res_ood = pd.DataFrame(rows).round(4)
res_ood.to_csv(f'{OUT}/05_감지.csv', index=False, encoding='utf-8-sig')
display(res_ood)

## 4. 1·2단계 기준값 정하기

In [ ]:
# 앱 경로 (judge.decide): 1등 확률 ≥ T_ONE → 1등만 (CLIP 안 봄) / ≥ T_KNOWN → 후보 합치기 + CLIP / 그 아래 → 2단계(CLIP만)
# → 처음 보는 음식이 'T_ONE 이상'이 되면 CLIP 확인 없이 틀린 표로 판정됨 = 가장 위험
p = lambda L: (L / T).softmax(1).max(1).values
def routes(L, y=None, t_one=0.8, t_known=0.3):
    q = p(L); r = {'1등만 (CLIP 안 봄)': (q >= t_one).float().mean().item(),
                   '합치기+CLIP': ((q >= t_known) & (q < t_one)).float().mean().item(),
                   '2단계 (CLIP만)': (q < t_known).float().mean().item()}
    if y is not None:
        m = q >= t_one
        r['1등만일 때 정확도'] = (L[m].argmax(1) == y[m]).float().mean().item() if m.any() else float('nan')
    return r

# A쪽으로 정하기: T_ONE = 처음 보는 음식의 '1등만' 비율이 5% 이하가 되는 가장 작은 값
#                T_KNOWN = 아는 음식의 90%가 1단계(표 사용)에 남는 값
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = pd.DataFrame([{'기준': t, '처음 보는 음식 ≥ 기준': (p(LnA) >= t).float().mean().item(),
                       '아는 음식 ≥ 기준': (p(LkA) >= t).float().mean().item(),
                       '아는 음식 ≥ 기준일 때 정확도': (LkA[p(LkA) >= t].argmax(1) == ykA[p(LkA) >= t]).float().mean().item()
                       if (p(LkA) >= t).any() else np.nan} for t in grid]).round(3)
sweep.to_csv(f'{OUT}/05_기준_탐색.csv', index=False, encoding='utf-8-sig')
display(sweep)
ok = sweep[sweep['처음 보는 음식 ≥ 기준'] <= 0.05]
T_ONE = float(ok.기준.min()) if len(ok) else 0.95
T_KNOWN = float(np.floor(np.quantile(p(LkA).numpy(), 0.10) * 20) / 20)    # 0.05 단위로 내림
T_KNOWN = min(T_KNOWN, T_ONE - 0.05)
print(f'추천: T_ONE = {T_ONE}, T_KNOWN = {T_KNOWN}  (지금 앱: 0.8, 0.3)')

# B쪽(기준 정할 때 안 쓴 음식·사진)으로 확인: 지금 기준 vs 추천 기준
rows = []
for name, (a, b) in {'지금 (0.8 / 0.3, 보정 전)': (0.8, 0.3), f'추천 ({T_ONE} / {T_KNOWN}, 보정 후)': (T_ONE, T_KNOWN)}.items():
    raw = name.endswith('보정 전)')
    LkB_, LnB_ = (LkB * T, LnB * T) if raw else (LkB, LnB)    # 보정 전 = 확률을 T로 안 나눈 것
    rows.append({'기준': name, '사진': '아는 음식 (B)', **routes(LkB_, ykB, a, b)})
    rows.append({'기준': name, '사진': '처음 보는 음식 (B, 기준 정할 때 안 쓴 37종)', **routes(LnB_, None, a, b)})
res_thr = pd.DataFrame(rows).round(3)
res_thr.to_csv(f'{OUT}/05_기준.csv', index=False, encoding='utf-8-sig')
display(res_thr)

cfg = {'temperature': round(T, 4), 'T_ONE': T_ONE, 'T_KNOWN': T_KNOWN,
       'note': '05_분류모델_보정_감지: T는 Food-101 test 절반으로 맞춤, 기준은 74종 중 절반으로 정함'}
json.dump(cfg, open(f'{CNX}/calibration.json', 'w'), ensure_ascii=False, indent=1)
print('저장:', f'{CNX}/calibration.json'); print(json.dumps(cfg, ensure_ascii=False))

## 끝 — 앱에 반영
1. 마지막 셀의 `calibration.json` 내용을 깃 `modeling/calibration.json`에 그대로 넣는다 → `modeling/model.py`가 T로 확률을 보정
2. `T_ONE`·`T_KNOWN`을 `knowledge/judge.py` 맨 위에 반영 (보정된 확률 기준)
3. 발표용: `표/05_보정.csv`(ECE 전/후), `05_감지.csv`(AUROC), `05_기준.csv`(처음 보는 음식이 CLIP 확인 없이 지나가는 비율 전/후), `05_보정_감지.png`